# TCRdist neighbor search figure

Panels A and B of the combined TCRdist/antibody figure. Reads
`../data/tcrdist_benchmark.csv`, produced by `benchmarks/04_tcrdist.ipynb`.

In [ ]:
import json 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyrepseq as prs

plt.style.use('bmh')

DATADIR = '../data/'

## Import TCR data

In [ ]:
df = pd.read_csv(f'{DATADIR}/tcrdist_benchmark.csv', index_col=0)
max_tcrdists = np.sort(df['max_tcrdist'].unique())
max_editss = sorted(df.loc[df['algorithm']=='symscan', 'max_edits'].dropna().unique().astype(int))

In [ ]:
def neighbors_by_threshold(sub):
    "{threshold: counts across samples}, samples in a consistent order"
    piv = sub.pivot_table(index='max_tcrdist', columns='sample', values='n_neighbors')
    return {int(dist): piv.loc[dist].to_numpy() for dist in max_tcrdists}

def times(sub):
    "one runtime per sample, deduplicated from the per-threshold rows"
    return sub.groupby('sample')['runtime_s'].first().sort_index().to_numpy()

exhaustive = df[df['algorithm']=='exhaustive']
neighbors_exhaustive = neighbors_by_threshold(exhaustive)
times_exhaustive = times(exhaustive)

results = {}
for max_edits in max_editss:
    sub = df[(df['algorithm']=='symscan') & (df['max_edits']==max_edits)]
    results[max_edits] = {'neighbors': neighbors_by_threshold(sub), 'times': times(sub)}

In [ ]:
for max_edits in max_editss:
    print(f'SymD {max_edits}: {times_exhaustive.mean()/results[max_edits]["times"].mean()}')

## Import antibody data

In [ ]:
with open(f'{DATADIR}/antibody_benchmark_results.json') as f:
    antibody = json.load(f)
# json keys are always strings, so cast max_edits keys back to int
times_symscan = {int(k): v for k, v in antibody['times_symscan'].items()}
fractions = {int(k): v for k, v in antibody['fractions'].items()}
time_exhaustive = antibody['time_exhaustive']
antibody_max_editss = antibody['max_editss']

## Plotting

In [ ]:
fig, axes_arr = plt.subplots(figsize=(3.42, 4.8), ncols=2, nrows=2)


axes = axes_arr[0]
for max_edits in max_editss:
    times_ = results[max_edits]['times']
    axes[0].plot(f'SymS {max_edits}', np.mean(times_), 'o', color=f'C{max_edits}')
    neighbors = results[max_edits]['neighbors']
    fraction = np.array([np.mean(neighbors[dist] / neighbors_exhaustive[dist]) for dist in max_tcrdists], dtype=float)
    print(max_tcrdists[np.argmax(fraction < 0.95)-1])
    axes[1].plot(max_tcrdists, fraction*100, label=f'd={max_edits}', color=f'C{max_edits}', zorder=10-max_edits)
axes[0].plot('Exh', np.mean(times_exhaustive), 'o', color='C0')
axes[1].axhline(100, color='C0', zorder=2)
axes[0].tick_params(axis='x', labelrotation=90)
axes[0].set_ylabel('Time in s')
axes[0].set_yscale('log')
axes[0].set_ylim(5e-2, 3e3)
axes[0].set_xlim(-0.5, 3.5)
axes[1].set_xlabel('TCRdist thresh.')
axes[1].set_ylabel('% Pairs')
axes[1].set_ylim(0, 105)
axes[1].set_xlim(0, 48)
axes[1].set_xticks(np.arange(0, 48, 12))

axes = axes_arr[1]
for max_edits in antibody_max_editss:
    axes[0].plot(f'SymS {max_edits}', np.sum(times_symscan[max_edits]), 'o', color=f'C{max_edits}')
    axes[1].plot([max_edits], [np.mean(fractions[max_edits])*100], 'o', color=f'C{max_edits}', zorder=3)
axes[0].plot('Exh', [time_exhaustive], 'o', color='C0')
axes[0].tick_params(axis='x', labelrotation=90)
axes[0].set_ylabel('Time in s')
axes[0].set_yscale('log')
axes[0].set_ylim(1e0, 5e2)
axes[0].set_xlim(-0.5, 3.5)
axes[1].axhline(y=100, color='C0', ls='-', zorder=2)
axes[1].set_xlabel('SymS edits')
axes[1].set_ylabel('% Pairs')
axes[1].set_ylim(80, 102)
axes[1].set_xlim(0.5, 3.5)


fig.tight_layout(w_pad=1.5, pad=0.0)
prs.plotting.label_axes(fig,xy=(-0.55, 1.0))
fig.savefig('figs/applications.svg')
fig.savefig('figs/applications.png', dpi=300)
fig.savefig('figs/applications.pdf')
